# TrueVoice— Data Pipeline

---

## Directory Structure

### Google Drive (/content/drive/MyDrive/2026truevoice/)

```
2026truevoice/
├── dataset/                          ← original archives (do not modify)
│   ├── ASVspoof2019.zip              ← original 25GB
│   ├── ASVspoof2019_LA_slim.tar.gz   ← train/dev/protocols only (slim version)
│   ├── ASVspoof2021_LA_eval.tar.gz   ← 2021 LA eval audio (7.2GB)
│   ├── keys.tar.gz                   ← 2021 eval score keys (slim)
│   └── LA-keys-full.tar.gz           ← full score keys
├── train.json                        ← train sample paths + labels (25,380) ✅
├── val.json                          ← val sample paths + labels (24,844) ✅
├── test_2021.json                    ← 2021 eval paths + labels (148,176) ✅
└── checkpoints/                      ← saved model weights
```

### Local /content/asvspoof/ (extracted per runtime, deleted on disconnect)

```
/content/asvspoof/
├── ASVspoof2019_LA_cm_protocols/
├── ASVspoof2019_LA_train/flac/
├── ASVspoof2019_LA_dev/flac/
├── ASVspoof2021_LA_eval/flac/
└── keys/LA/CM/trial_metadata.txt
```

---

## Dataset Info

| Split | Dataset | real | fake | ratio |
|-------|---------|------|------|-------|
| Train | ASVspoof 2019 LA train | 2,580 | 22,800 | 1:8 |
| Val   | ASVspoof 2019 LA dev   | 2,548 | 22,296 | 1:8 |
| Test  | ASVspoof 2021 LA eval  | — | — | 148,176 total |

- Sampling rate: 16kHz
- Max clip length: 29s (Gemma 4 audio_tower limit is 30s)
- Codec simulation applied to train/val (removes studio domain fingerprint)
- Test is used as-is (2021 LA already contains real codec/transmission effects)

---

## Execution Order (after reconnecting runtime)

Run cells 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 in order (~30-40 minutes)

Since train.json / val.json / test_2021.json are already saved to Drive,
you can skip cell 8 and load them directly:

    import json
    with open(f'{DRIVE_BASE}/train.json') as f: train_samples = json.load(f)
    with open(f'{DRIVE_BASE}/val.json') as f:   val_samples   = json.load(f)
    with open(f'{DRIVE_BASE}/test_2021.json') as f: test_samples = json.load(f)

---

## Model Context

Model: google/gemma-4-e4b-it
Task: Use Gemma 4 E4B's audio_tower as a feature extractor for real/fake binary classification
Training method: Linear Probe (audio_tower frozen, classification head only)
Compute: Google Colab Pro (A100)
Demo: Gradio (web-based)

[Audio Paths]
Train audio     : /content/asvspoof/LA/LA/ASVspoof2019_LA_train/flac/
Train protocol  : /content/asvspoof/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt
Val audio       : /content/asvspoof/LA/LA/ASVspoof2019_LA_dev/flac/
Val protocol    : /content/asvspoof/LA/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt
Test audio      : /content/asvspoof/ASVspoof2021_LA_eval/flac/
Test keys       : /content/asvspoof/keys/LA/CM/trial_metadata.txt
Test key format : SPEAKER_ID FILENAME CODEC LANG SYSTEM_ID KEY TRIM SUBSET
                  KEY = parts[5] (bonafide/spoof), use only SUBSET == 'eval' (parts[7])

[JSON Index on Drive]
/content/drive/MyDrive/2026truevoice/train.json      (25,380 samples)
/content/drive/MyDrive/2026truevoice/val.json        (24,844 samples)
/content/drive/MyDrive/2026truevoice/test_2021.json  (148,176 samples)
Format: [{"audio_path": "...", "label": "real" or "fake"}, ...]

[Preprocessing]
- Audio: 16kHz, mono, max 29 seconds
- Codec simulation applied to train/val only (prevents shortcut learning)
  codec_simulate(): 16kHz → 8kHz → 16kHz resample
- Test used as-is (2021 LA already has codec effects)
- DataCollator pattern: load audio per batch + pad (avoids RAM overflow)
- TrainingArguments must include remove_unused_columns=False

[Model Architecture]
class AudioDeepfakeClassifier(nn.Module):
    audio_tower : Gemma4 E4B audio_tower (frozen)
    classifier  : Linear(hidden_size, 256) -> GELU -> Dropout(0.3) -> Linear(256, 2)
    pooling     : last_hidden_state.mean(dim=1)
    input       : input_features (mel spectrogram)
    output      : SequenceClassifierOutput

[Training Config]
learning_rate             : 1e-3
epochs                    : 3
per_device_train_batch_size: 2
gradient_accumulation_steps: 8  (effective batch size 16)
bf16                      : False (float32 required to avoid NaN in audio_tower)
metric                    : f1_macro
checkpoints               : /content/drive/MyDrive/2026truevoice/checkpoints/

[Evaluation Metrics]
Primary   : EER (Equal Error Rate, lower is better)
Secondary : F1-macro, classification_report
classification_report(y_true, y_pred, labels=[0,1], target_names=['real','fake'])


## Cell 1: Install Packages

In [ ]:
%%capture
!pip install librosa soundfile torchaudio scikit-learn scipy datasets transformers
print('Packages installed')

## Cell 2: Mount Drive + Extract Archives

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

DRIVE_DATASET = '/content/drive/MyDrive/2026truevoice/dataset'
LOCAL_BASE    = '/content/asvspoof'
DATASET_DIR   = '/content/dataset'

os.makedirs(LOCAL_BASE,  exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)


def copy_and_extract(zip_name, marker):
    """Copy archive from Drive to local storage and extract.
    Skips extraction if the marker directory already exists.
    """
    marker_path = os.path.join(LOCAL_BASE, marker)
    if os.path.exists(marker_path):
        print(f'  Already extracted: {marker}')
        return
    src = f'{DRIVE_DATASET}/{zip_name}'
    dst = f'/content/{zip_name}'
    print(f'  Copying: {zip_name} ...')
    os.system(f'cp "{src}" "{dst}"')
    print(f'  Extracting...')
    if zip_name.endswith('.zip'):
        os.system(f'unzip -q "{dst}" -d "{LOCAL_BASE}/"')
    elif zip_name.endswith(('.tar.gz', '.tgz')):
        os.system(f'tar -xzf "{dst}" -C "{LOCAL_BASE}/"')
    os.system(f'rm "{dst}"')
    print(f'  Done: {marker}')


print('=== Extracting archives ===')
copy_and_extract('ASVspoof2019_LA_slim.tar.gz', 'ASVspoof2019_LA_train')
copy_and_extract('keys.tar.gz',                 'keys')
copy_and_extract('ASVspoof2021_LA_eval.tar.gz', 'ASVspoof2021_LA_eval')
print('\n=== Done ===')

## Cell 3: Set & Verify Paths

In [ ]:
import os

BASE_2019       = '/content/asvspoof/LA/LA'
TRAIN_PROTO     = f'{BASE_2019}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
VAL_PROTO       = f'{BASE_2019}/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
TRAIN_AUDIO_DIR = f'{BASE_2019}/ASVspoof2019_LA_train/flac'
VAL_AUDIO_DIR   = f'{BASE_2019}/ASVspoof2019_LA_dev/flac'
BASE_2021_AUDIO = '/content/asvspoof/ASVspoof2021_LA_eval/flac'
BASE_2021_KEY   = '/content/asvspoof/keys/LA/CM/trial_metadata.txt'

print('=== Path verification ===')
for label, path in [
    ('TRAIN_PROTO',     TRAIN_PROTO),
    ('VAL_PROTO',       VAL_PROTO),
    ('TRAIN_AUDIO_DIR', TRAIN_AUDIO_DIR),
    ('VAL_AUDIO_DIR',   VAL_AUDIO_DIR),
    ('BASE_2021_AUDIO', BASE_2021_AUDIO),
    ('BASE_2021_KEY',   BASE_2021_KEY),
]:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'  {status} {label}: {path}')

## Cell 4: Parse ASVspoof 2019 Protocol

In [ ]:
import os

def parse_2019_protocol(proto_path, audio_dir):
    """Parse ASVspoof 2019 protocol file into a list of {audio_path, label} dicts.

    Protocol format: SPEAKER_ID FILENAME _ SYSTEM_ID KEY
    KEY: 'bonafide' -> real, 'spoof' -> fake

    Args:
        proto_path : path to .trn or .trl protocol file
        audio_dir  : directory containing .flac audio files

    Returns:
        list of {audio_path: str, label: 'real' | 'fake'}
    """
    samples  = []
    missing  = 0

    with open(proto_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            filename = parts[1]
            key      = parts[4]   # 'bonafide' or 'spoof'
            label    = 'real' if key == 'bonafide' else 'fake'
            audio_path = os.path.join(audio_dir, f'{filename}.flac')

            if not os.path.exists(audio_path):
                missing += 1
                continue

            samples.append({'audio_path': audio_path, 'label': label})

    print(f'  Parsed: {len(samples)} samples | Missing files: {missing}')
    return samples

print('parse_2019_protocol defined')

## Cell 5: Parse ASVspoof 2021 LA eval Keys

In [ ]:
def parse_2021_la_keys(key_path, audio_dir):
    """Parse ASVspoof 2021 LA eval trial metadata into {audio_path, label} dicts.

    Key file format: SPEAKER_ID FILENAME CODEC LANG SYSTEM_ID KEY TRIM SUBSET
    - KEY    = parts[5]: 'bonafide' -> real, 'spoof' -> fake
    - SUBSET = parts[7]: only use rows where SUBSET == 'eval'

    Args:
        key_path  : path to trial_metadata.txt
        audio_dir : directory containing .flac audio files

    Returns:
        list of {audio_path: str, label: 'real' | 'fake'}
    """
    samples  = []
    missing  = 0
    skipped  = 0

    with open(key_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 8:
                continue

            subset = parts[7]
            if subset != 'eval':       # use eval subset only
                skipped += 1
                continue

            filename   = parts[1]
            key        = parts[5]      # 'bonafide' or 'spoof'
            label      = 'real' if key == 'bonafide' else 'fake'
            audio_path = os.path.join(audio_dir, f'{filename}.flac')

            if not os.path.exists(audio_path):
                missing += 1
                continue

            samples.append({'audio_path': audio_path, 'label': label})

    print(f'  Parsed: {len(samples)} samples | Missing: {missing} | Non-eval skipped: {skipped}')
    return samples

print('parse_2021_la_keys defined')

## Cell 6: Define Dataset & DataCollator

In [ ]:
import torch
import torchaudio
import librosa
from dataclasses import dataclass
from typing import List, Dict
from datasets import Dataset


def codec_simulate(audio_np, sr=16000):
    """Simulate phone codec: downsample to 8kHz then upsample back to 16kHz.
    Bridges the domain gap between studio-quality 2019 audio
    and real-world phone call conditions in 2021.
    """
    waveform = torch.from_numpy(audio_np).unsqueeze(0).float()
    down = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=8000)
    up   = torchaudio.functional.resample(down,     orig_freq=8000, new_freq=sr)
    return up.squeeze(0).numpy()


def load_audio(path, sr=16000, max_sec=29.0, apply_codec=False):
    """Load audio, clip to max_sec, and optionally apply codec simulation."""
    audio, _ = librosa.load(path, sr=sr, mono=True)
    max_samples = int(max_sec * sr)
    if len(audio) > max_samples:
        audio = audio[:max_samples]
    if apply_codec:
        audio = codec_simulate(audio, sr=sr)
    return audio


def make_clf_dataset(samples):
    """Store only file paths and labels (not audio).
    Audio is loaded per batch by the collator to avoid RAM overflow.
    """
    return Dataset.from_dict({
        'audio_path': [s['audio_path'] for s in samples],
        'labels':     [0 if s['label'] == 'real' else 1 for s in samples],
    })


@dataclass
class AudioDataCollator:
    """On-the-fly audio loader: loads audio per batch, converts to mel spectrogram, pads.

    Args:
        processor   : Gemma4 AutoProcessor
        apply_codec : True  for train/val (apply codec simulation to 2019 studio audio)
                      False for test  (2021 LA already has real codec effects)
    """
    processor: object
    apply_codec: bool = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        batch_features, batch_labels = [], []

        for item in features:
            try:
                audio  = load_audio(item['audio_path'], apply_codec=self.apply_codec)
                inputs = self.processor(
                    text='<audio>',
                    audio=audio,
                    sampling_rate=16000,
                    return_tensors='pt',
                )
                if 'input_features' not in inputs:
                    continue
                batch_features.append(inputs['input_features'][0])
                batch_labels.append(item['labels'])
            except Exception:
                continue  # skip corrupted or missing files silently

        if not batch_features:
            raise RuntimeError('No valid audio samples in batch.')

        # Gemma4 processor returns shape (time, mel=128) — pad along time axis
        max_len = max(f.shape[0] for f in batch_features)
        padded  = torch.stack([
            torch.nn.functional.pad(f, (0, 0, 0, max_len - f.shape[0]))
            for f in batch_features
        ])

        return {
            'input_features': padded.float(),
            'labels': torch.tensor(batch_labels, dtype=torch.long),
        }

print('Dataset and DataCollator defined')

## Cell 7: Define Streaming Evaluation Function (2021 eval)

In [ ]:
def evaluate_2021(model, processor, samples, device='cuda', batch_size=8):
    """Streaming batch evaluation on ASVspoof 2021 LA eval.

    Audio is not preloaded into memory — processed per batch for memory safety.
    No codec simulation applied — 2021 LA already contains real codec effects.
    loaded_samples tracks which samples were successfully loaded to prevent misalignment.

    Args:
        model      : trained classifier
        processor  : Gemma4 AutoProcessor
        samples    : list of {audio_path, label} dicts
        device     : 'cuda'
        batch_size : samples per batch

    Returns:
        dict with eer, y_true, y_pred
    """
    from sklearn.metrics import classification_report, roc_curve
    from scipy.optimize import brentq
    from scipy.interpolate import interp1d

    model.eval()
    y_true, y_pred, y_score = [], [], []

    for i in range(0, len(samples), batch_size):
        batch = samples[i : i + batch_size]

        loaded_features = []
        loaded_samples  = []

        for sample in batch:
            try:
                # No codec simulation — 2021 eval already has real codec effects
                audio = load_audio(sample['audio_path'], apply_codec=False)
                inputs = processor(
                    text='<audio>',
                    audio=audio,
                    sampling_rate=16000,
                    return_tensors='pt',
                )
                loaded_features.append(inputs['input_features'][0])
                loaded_samples.append(sample)
            except Exception:
                continue

        if not loaded_features:
            continue

        # Pad along time axis (dim=0)
        max_len = max(f.shape[0] for f in loaded_features)
        padded  = torch.stack([
            torch.nn.functional.pad(f, (0, 0, 0, max_len - f.shape[0]))
            for f in loaded_features
        ]).to(device)

        with torch.no_grad():
            output = model(input_features=padded)
            probs  = torch.softmax(output.logits, dim=-1)
            preds  = output.logits.argmax(-1).cpu().tolist()
            scores = probs[:, 1].cpu().tolist()   # fake class probability

        for j, sample in enumerate(loaded_samples):
            y_true.append(0 if sample['label'] == 'real' else 1)
            y_pred.append(preds[j])
            y_score.append(scores[j])

        if i % (batch_size * 50) == 0:
            print(f'  Progress: {i}/{len(samples)}')

    # Compute EER via ROC curve interpolation
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1)
    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)

    print(f'\nASVspoof 2021 LA eval Results')
    print(f'EER: {eer*100:.2f}%  (lower is better)')
    print(f'\n{classification_report(y_true, y_pred, labels=[0,1], target_names=["real","fake"])}')

    return {'eer': eer, 'y_true': y_true, 'y_pred': y_pred}

print('evaluate_2021 defined')

## Cell 8: Run — Parse Protocols → Build Datasets → Save JSON

In [ ]:
import json

print('=== Parsing ASVspoof 2019 ===')
train_samples = parse_2019_protocol(TRAIN_PROTO, TRAIN_AUDIO_DIR)
val_samples   = parse_2019_protocol(VAL_PROTO,   VAL_AUDIO_DIR)

print()
for name, s in [('train', train_samples), ('val', val_samples)]:
    r = sum(1 for x in s if x['label'] == 'real')
    f = sum(1 for x in s if x['label'] == 'fake')
    print(f'  {name}: real {r} / fake {f} (ratio 1:{f//r if r else "?"})')

print('\n=== Building datasets ===')
train_dataset = make_clf_dataset(train_samples)
val_dataset   = make_clf_dataset(val_samples)
print(f'  train: {len(train_dataset)}, val: {len(val_dataset)}')

print('\n=== Parsing ASVspoof 2021 LA eval ===')
test_samples = parse_2021_la_keys(BASE_2021_KEY, BASE_2021_AUDIO)

print('\n=== Done ===')

In [ ]:
# Save parsed samples to Drive as JSON
# Run this once — subsequent runs can load these files directly
import json

DRIVE_BASE = '/content/drive/MyDrive/2026truevoice'

with open(f'{DRIVE_BASE}/train.json', 'w') as f:
    json.dump(train_samples, f)
with open(f'{DRIVE_BASE}/val.json', 'w') as f:
    json.dump(val_samples, f)
with open(f'{DRIVE_BASE}/test_2021.json', 'w') as f:
    json.dump(test_samples, f)

print(f'train     : {len(train_samples):,} samples')
print(f'val       : {len(val_samples):,} samples')
print(f'test_2021 : {len(test_samples):,} samples')
print('Saved to Drive successfully')

## Cell 9: Trainer Config Reference

Copy the following into the fine-tuning notebook:

    train_collator = AudioDataCollator(processor=processor, apply_codec=True)

    training_args = TrainingArguments(
        output_dir='/content/drive/MyDrive/2026truevoice/checkpoints',
        remove_unused_columns=False,   # required — keeps audio_path column for collator
        num_train_epochs=3,
        learning_rate=1e-3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=50,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        bf16=False,                    # float32 required — bfloat16 causes NaN in audio_tower
        dataloader_num_workers=0,      # required — multiprocessing causes collator caching issues
        logging_steps=50,
        report_to='none',
    )

    trainer = Trainer(
        model=clf_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=train_collator,
        compute_metrics=compute_metrics,
    )

    # Evaluation (after training)
    results = evaluate_2021(clf_model, processor, test_samples)
